# PNPL 2026 — submission smoke test

Tiny end-to-end Colab notebook that:
1. installs `pnpl` (from this branch) and the latest `kaggle` CLI,
2. builds a randomly-initialised submission CSV in the required format,
3. uploads it to the PNPL 2026 competition on Kaggle.

The submission format is `index, <50 primary-vocab probs>, <50 moses-vocab probs (prefixed `moses_`)>`. Only the primary distribution is scored (balanced accuracy on argmax); the moses block rides along for downstream use.

**Auth:** the submission step needs a Kaggle API token. Generate one at <https://www.kaggle.com/settings/api> (the modern token looks like `KGAT_…`). Don't paste it into the notebook source — keep it in a Colab secret or the runtime env.

## 1. Install dependencies

In [ ]:
# Pin to >=2.0 for KAGGLE_API_TOKEN (KGAT_…) support; needs Python >= 3.11 (Colab default in 2026).
%pip install -q --upgrade "kaggle>=2.0" numpy pandas
# Install pnpl from GitHub. Replace the URL with a local install if you're hacking on the package.
%pip install -q git+https://github.com/neural-processing-lab/pnpl-public.git

## 2. Configure Kaggle credentials

On Colab, prefer storing the token as a secret. Two common patterns:

**Option A — Colab user secret (recommended).** Click the 🔑 icon in the left sidebar, add a secret named `KAGGLE_API_TOKEN`, enable access for this notebook, then run the cell below.

**Option B — paste at runtime.** Run `getpass.getpass()` and paste; the token never enters the notebook source.

Either way, the token is exported as `KAGGLE_API_TOKEN` for the rest of the session.

In [ ]:
import os

if not os.environ.get("KAGGLE_API_TOKEN"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token (KGAT_…): ").strip()

assert os.environ["KAGGLE_API_TOKEN"].startswith("KGAT_"), \
    "Expected a modern Kaggle API token starting with 'KGAT_'. Generate one at https://www.kaggle.com/settings/api"
print("token configured (length =", len(os.environ["KAGGLE_API_TOKEN"]), ")")

## 3. Build a submission CSV

We need one row per holdout index. Here we fake `N` rows with uniform-random probability distributions over both vocabularies. Replace `primary_probs` / `secondary_probs` with your model outputs.

In [ ]:
import numpy as np
from pnpl.competition import PRIMARY_VOCAB, SECONDARY_VOCAB, write_submission

print(f"primary vocab ({len(PRIMARY_VOCAB)}): {PRIMARY_VOCAB[:5]} ... {PRIMARY_VOCAB[-3:]}")
print(f"moses vocab   ({len(SECONDARY_VOCAB)}): {SECONDARY_VOCAB[:5]} ... {SECONDARY_VOCAB[-3:]}")

# TODO: replace with the real holdout index list and your model's per-row probabilities.
N = 100
rng = np.random.default_rng(2026)
indices = np.arange(N)

def random_probs(n_rows, vocab_size, seed):
    g = np.random.default_rng(seed)
    logits = g.standard_normal((n_rows, vocab_size))
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs /= probs.sum(axis=1, keepdims=True)
    return probs

primary_probs   = random_probs(N, len(PRIMARY_VOCAB),   seed=1)
secondary_probs = random_probs(N, len(SECONDARY_VOCAB), seed=2)

csv_path = write_submission(
    "submission.csv",
    indices=indices,
    primary_probs=primary_probs,
    secondary_probs=secondary_probs,
)
print(f"wrote {csv_path}")

# Peek at the first 3 columns of the first 3 rows so you can eyeball the format.
import pandas as pd
pd.read_csv(csv_path).iloc[:3, :4]

## 4. Submit to Kaggle

In [ ]:
from pnpl.competition import submit_to_kaggle

result = submit_to_kaggle(
    csv_path,
    competition="pnpl-internal-testing",
    message="colab smoke test — uniform random probs",
)
print("stdout:", result.stdout)
print("stderr:", result.stderr)

## 5. (Optional) Check submission status

After submitting, Kaggle takes a few seconds to score. List your recent submissions to see the leaderboard score.

In [ ]:
!kaggle competitions submissions -c pnpl-internal-testing | head -10